In [ ]:
%%capture
# uv sync --group develop --group lint --group test --group notebook
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dj_notebook import activate
from django_pandas.io import read_frame

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from edc_pdutils.dataframes import get_eos, get_subject_visit
from edc_randomization.models import RandomizationList
from meta_prn.models import EndOfStudy
from meta_lists.models import OffstudyReasons
from meta_screening.models import SubjectScreening
from django_crypto_fields.fields import EncryptedCharField
from clinicedc_constants import YES
from meta_screening.eligibility import EligibilityPartOne, EligibilityPartTwo, EligibilityPartThreePhaseThree
from clinicedc_constants import NO


In [ ]:
df_rando = read_frame(RandomizationList.objects.values("subject_identifier", "sid", "assignment", "randomizer_name", "allocated_site").all(), verbose=False)
for col in df_rando.select_dtypes(include=['datetime64[ns, UTC]']).columns:
    df_rando[col] = df_rando[col].dt.tz_localize(None)

In [ ]:
# df_eos = read_frame(EndOfStudy.objects.all(), verbose=False)
# read_frame(OffstudyReasons.objects.all(), verbose=False)
df_eos = get_eos("meta_prn.endofstudy", all_fields=True)
for col in df_eos.select_dtypes(include=['datetime64[ns, UTC]']).columns:
    df_eos[col] = df_eos[col].dt.tz_localize(None)

In [ ]:
df_visit = get_subject_visit(model="meta_subject.subjectvisit")

In [ ]:
fields = [f.name for f in SubjectScreening._meta.get_fields() if not isinstance(f, EncryptedCharField)]
df_screening = read_frame(SubjectScreening.objects.values(*fields).all(), verbose=False)
for col in df_screening.select_dtypes(include=['datetime64[ns, UTC]']).columns:
    df_screening[col] = df_screening[col].dt.tz_localize(None)

In [ ]:
df_screening["eligible_part_one"] = np.where(df_screening["eligible_part_one"]==YES, 1, 0)
df_screening["eligible_part_two"] = np.where(df_screening["eligible_part_two"]==YES, 1, 0)
df_screening["eligible_part_three"] = np.where(df_screening["eligible_part_three"]==YES, 1, 0)
df_screening["eligible"] = df_screening["eligible"].astype(int)

for col in df_screening.columns:
    if col.startswith("calculated") or col.startswith("converted"):
        df_screening[col] = df_screening[col].astype("Float64")



In [ ]:
# 10576
print(f'Screened: {len(df_screening)}')

# 1728
print(f'Eligible: {len(df_screening[df_screening["eligible"]==1])}')

# 1691
print(f'Consented: {len(df_screening[df_screening["subject_identifier"].str.startswith("105-")])}')


In [ ]:
df_screening[((df_screening["eligible_part_one"]==0) | ((df_screening["eligible_part_one"]==1) & (df_screening["eligible_part_two"]==0)))].screening_identifier.nunique()


In [ ]:

class P1(EligibilityPartOne):
    def __init__(self, **kwargs):
        pass
    def _assess_eligibility(self):
        pass

for col in P1().get_required_fields():
    if col in ["age_in_years", "gender"]:
        continue
    if col in ["meta_phase_two", "pregnant"]:
        print(col, df_screening[df_screening[col]==YES].subject_identifier.nunique())
        continue
    print(col, df_screening[df_screening[col]==NO].subject_identifier.nunique())


In [ ]:
class P2(EligibilityPartTwo):
    def __init__(self, **kwargs):
        pass
    def _assess_eligibility(self):
        pass

for col in P2().get_required_fields():
    if col in ["appt_datetime"]:
        continue
    print(col, df_screening[df_screening[col]==YES].subject_identifier.nunique())


In [ ]:

class P3(EligibilityPartThreePhaseThree):
    def __init__(self, **kwargs):
        pass
    def _assess_eligibility(self):
        pass

for col in P3().get_required_fields():
    if col in ["creatinine_units", "creatinine_value"]:
        continue
    print(col, df_screening[~df_screening[col].isna()].subject_identifier.nunique())


In [ ]:
df_scr = df_screening.copy()

In [ ]:
df_scr["reasons_ineligible"] = df_scr.reasons_ineligible.str.split("|")
df_long = df_scr.explode("reasons_ineligible")
df_long["reasons_ineligible"] = df_long["reasons_ineligible"].str.strip()
df_long[(df_long["eligible"]==0) & (df_long["part_three_report_datetime"].isna())].reasons_ineligible.value_counts()

## pivot on the split values of reasons_ineligible
# df_pivot = df_scr["reasons_ineligible"].str.get_dummies(sep="|")
# df_scr = pd.concat([df_scr["screening_identifier"], df_pivot], axis=1)
# pivot_cols = list(df_pivot.columns)
# df_scr[["screening_identifier", *pivot_cols]]

In [ ]:
df_long[(df_long["eligible_part_one"]==0) | ((df_long["eligible_part_one"]==1) & (df_long["eligible_part_two"]==0))].reasons_ineligible.value_counts()

In [ ]:
# screened, ineligible before glucose
df_long[(df_long["eligible"]==0) & (df_long["part_three_report_datetime"].isna())]

In [ ]:
# screened, glucose, but ineligible

# IFT/OGTT-> inclusion_a=NO & inclusion_b=NO

print(df_long[(df_long["eligible"]==0) & (~df_long["part_three_report_datetime"].isna())].screening_identifier.nunique())

df_long[(df_long["eligible"]==0) & (~df_long["part_three_report_datetime"].isna())].reasons_ineligible.value_counts()


In [ ]:
df_long[df_long["reasons_ineligible"]=="IFT/OGTT"].inclusion_a.value_counts()
df_long[df_long["reasons_ineligible"]=="IFT/OGTT"].inclusion_b.value_counts()


df_scr[df_scr["reasons_ineligible"].apply(lambda x: "IFT/OGTT" in x)][[col for col in df_scr.columns if col.startswith("converted") or col.startswith("calculated") or col.startswith("inclusion")]]



In [ ]:
# screened, glucose and eligible
df_long[(df_long["eligible"]==1) & (~df_long["part_three_report_datetime"].isna())].screening_identifier.nunique()
